# Real-Estate Classifier - Inference Walkthrough

Notebook para probar inferencia local con checkpoint y endpoint FastAPI.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

In [ ]:
from PIL import Image
import requests

from real_estate_ml.inference.predictor import Predictor

model_path = ROOT / "artifacts" / "best_model.pth"
predictor = Predictor(
    checkpoint_path=model_path,
    backbone="efficientnet_b0",
    num_classes=15,
    device="cuda",
)

print("Model loaded from:", model_path)

In [ ]:
# Busca una imagen de ejemplo en test
sample_candidates = list((ROOT / "data" / "processed" / "test").glob("*/*.jpg"))
if not sample_candidates:
    sample_candidates = list((ROOT / "data" / "processed" / "test").glob("*/*.png"))

sample_path = sample_candidates[0]
sample_image = Image.open(sample_path).convert("RGB")
print("Sample image:", sample_path)
sample_image

In [ ]:
# Predicción local (sin API)
local_preds = predictor.predict(sample_image, top_k=3)
local_preds

In [ ]:
# Predicción vía API (requiere uvicorn corriendo)
api_url = "http://127.0.0.1:8000/predict"
with open(sample_path, "rb") as file:
    response = requests.post(api_url, files={"file": (sample_path.name, file, "image/jpeg")}, timeout=30)

response.status_code, response.json()